In [ ]:
import os
import math
import warnings
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from tqdm import tqdm

warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as transforms
from torchvision.utils import make_grid

from skimage.metrics import peak_signal_noise_ratio, structural_similarity

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

CONFIG = {
    'dataset_paths': [
        '/kaggle/input/datasets/denislukovnikov/celebahq256-images-only/celebahq256_imgs/train',
    ],
    'image_size': 128,
    'batch_size': 4,
    'num_workers': 2,

    'num_timesteps': 300,
    'base_channels': 64,
    'time_dim': 256,
    'channel_mult': [1, 2, 2],
    'num_res_blocks': 2,
    'dropout': 0.1,

    'lr': 1e-4,
    'num_epochs': 50,
    'gradient_clip': 1.0,
    'warmup_steps': 1000,
    'ema_decay': 0.9999,
    'gradient_accumulation_steps': 2,

    'schedule': 'cosine',
    'beta_start': 1e-4,
    'beta_end': 0.02,

    'ddim_steps': 50,
    'ddim_eta': 0.0,

    'output_dir': './ddpm_outputs',
    'ckpt_dir': './ddpm_checkpoints',
    'save_every': 10,
    'eval_every': 5,

    'seed': 42,
    'mixed_precision': True,
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)
os.makedirs(CONFIG['ckpt_dir'], exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

class ImageDataset(Dataset):
    def __init__(self, paths, image_size=128):
        self.image_size = image_size
        self.files = []

        for path in paths:
            path_obj = Path(path)
            if path_obj.exists():
                for ext in ['*.jpg', '*.jpeg', '*.png', '*.webp']:
                    self.files.extend(list(path_obj.rglob(ext)))
                if self.files:
                    print(f"Found {len(self.files)} images")
                    break

        if not self.files:
            print("Using synthetic data")
            self.use_synthetic = True
            self.num_synthetic = 1000
        else:
            self.use_synthetic = False

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.CenterCrop(image_size),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self):
        return self.num_synthetic if self.use_synthetic else len(self.files)

    def __getitem__(self, idx):
        if self.use_synthetic:
            return torch.randn(3, self.image_size, self.image_size) * 0.5
        try:
            return self.transform(Image.open(self.files[idx]).convert('RGB'))
        except Exception:
            return torch.randn(3, self.image_size, self.image_size)

def inverse_transform(tensor):
    tensor = tensor.cpu().detach()
    tensor = torch.clamp(tensor, -1, 1)
    tensor = (tensor + 1) / 2
    tensor = torch.clamp(tensor, 0, 1)
    
    if tensor.dim() == 3:
        return tensor.permute(1, 2, 0).numpy()
    elif tensor.dim() == 4:
        return tensor.permute(0, 2, 3, 1).numpy()
    return tensor.numpy()

class NoiseScheduler:
    def __init__(self, num_timesteps=300, beta_start=1e-4, beta_end=0.02, schedule='cosine'):
        self.num_timesteps = num_timesteps

        if schedule == 'linear':
            betas = torch.linspace(beta_start, beta_end, num_timesteps)
        else:
            s = 0.008
            x = torch.linspace(0, num_timesteps, num_timesteps + 1)
            alphas_bar_full = torch.cos(((x / num_timesteps) + s) / (1 + s) * math.pi / 2) ** 2
            alphas_bar_full = alphas_bar_full / alphas_bar_full[0]
            betas = 1 - (alphas_bar_full[1:] / alphas_bar_full[:-1])
            betas = torch.clamp(betas, 1e-4, 0.9999)

        alphas = 1.0 - betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        self.betas = betas
        self.alphas = alphas
        self.alphas_bar = alphas_bar
        self.alphas_bar_prev = F.pad(alphas_bar[:-1], (1, 0), value=1.0)

    def extract(self, a, t, x_shape):
        out = a.to(t.device)[t]
        return out.reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab  = torch.sqrt(self.extract(self.alphas_bar, t, x0.shape))
        sqrt_1mab = torch.sqrt(1 - self.extract(self.alphas_bar, t, x0.shape))
        return sqrt_ab * x0 + sqrt_1mab * noise, noise

class EMA:
    def __init__(self, model, decay=0.9999):
        self.model  = model
        self.decay  = decay
        self.shadow = {n: p.data.clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (1 - self.decay) * param.data + self.decay * self.shadow[name]

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.backup:
                param.data = self.backup[name]

class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half = self.dim // 2
        emb  = math.log(10000) / (half - 1)
        emb  = torch.exp(torch.arange(half, device=time.device) * -emb)
        emb  = time[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.1):
        super().__init__()
        self.norm1     = nn.GroupNorm(min(32, in_ch), in_ch)
        self.conv1     = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2     = nn.GroupNorm(min(32, out_ch), out_ch)
        self.conv2     = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act       = nn.SiLU()
        self.dropout   = nn.Dropout(dropout)
        self.shortcut  = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.time_proj(self.act(t_emb))[:, :, None, None]
        h = self.conv2(self.dropout(self.act(self.norm2(h))))
        return h + self.shortcut(x)

class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=256,
                 channel_mult=(1, 2, 2), num_res_blocks=2, dropout=0.1):
        super().__init__()
        self.num_res_blocks = num_res_blocks

        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbedding(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.conv_in = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        self.enc_blocks = nn.ModuleList()
        self.enc_downsamplers = nn.ModuleList()
        skip_ch_list = []

        ch = base_channels
        for i, mult in enumerate(channel_mult):
            out_ch = base_channels * mult
            level = nn.ModuleList()
            for _ in range(num_res_blocks):
                level.append(ResidualBlock(ch, out_ch, time_dim, dropout))
                skip_ch_list.append(out_ch)
                ch = out_ch
            self.enc_blocks.append(level)
            self.enc_downsamplers.append(
                Downsample(ch) if i < len(channel_mult) - 1 else nn.Identity()
            )

        self.bottleneck = nn.ModuleList([
            ResidualBlock(ch, ch, time_dim, dropout),
            ResidualBlock(ch, ch, time_dim, dropout),
        ])

        self.dec_blocks = nn.ModuleList()
        self.dec_upsamplers = nn.ModuleList()

        rev_skips = list(reversed(skip_ch_list))
        skip_idx = 0

        for i, mult in enumerate(reversed(channel_mult)):
            out_ch = base_channels * mult
            level = nn.ModuleList()
            for _ in range(num_res_blocks):
                skip_w = rev_skips[skip_idx]
                skip_idx += 1
                level.append(ResidualBlock(ch + skip_w, out_ch, time_dim, dropout))
                ch = out_ch
            self.dec_blocks.append(level)
            self.dec_upsamplers.append(
                Upsample(ch) if i < len(channel_mult) - 1 else nn.Identity()
            )

        self.conv_out = nn.Sequential(
            nn.GroupNorm(min(32, ch), ch),
            nn.SiLU(),
            nn.Conv2d(ch, in_channels, 3, padding=1),
        )

    def forward(self, x, time):
        t = self.time_mlp(time)

        h = self.conv_in(x)
        skips = []
        for level, down in zip(self.enc_blocks, self.enc_downsamplers):
            for block in level:
                h = block(h, t)
                skips.append(h)
            h = down(h)

        for block in self.bottleneck:
            h = block(h, t)

        for level, up in zip(self.dec_blocks, self.dec_upsamplers):
            for block in level:
                skip = skips.pop()
                if h.shape[-2:] != skip.shape[-2:]:
                    h = F.interpolate(h, size=skip.shape[-2:],
                                      mode='bilinear', align_corners=False)
                h = torch.cat([h, skip], dim=1)
                h = block(h, t)
            h = up(h)

        return self.conv_out(h)

class DDPM(nn.Module):
    def __init__(self, unet, scheduler, img_size=128):
        super().__init__()
        self.unet = unet
        self.scheduler = scheduler
        self.img_size = img_size
        self.num_timesteps = scheduler.num_timesteps

    def loss(self, x0, t):
        x_t, noise = self.scheduler.q_sample(x0, t)
        return F.mse_loss(self.unet(x_t, t), noise)

    @torch.no_grad()
    def ddim_sample(self, batch_size=1, num_steps=50, eta=0.0):
        self.unet.eval()
        dev = next(self.unet.parameters()).device
        steps = torch.linspace(self.num_timesteps - 1, 0, num_steps,
                               dtype=torch.long, device=dev)
        x = torch.randn(batch_size, 3, self.img_size, self.img_size, device=dev)

        for i, t in enumerate(steps):
            t_b = t.expand(batch_size)
            pred = self.unet(x, t_b)
            ab = self.scheduler.extract(self.scheduler.alphas_bar, t_b, x.shape)
            x0p = torch.clamp((x - (1 - ab).sqrt() * pred) / ab.sqrt(), -1, 1)

            if i < len(steps) - 1:
                ab_n = self.scheduler.extract(self.scheduler.alphas_bar,
                                              steps[i + 1].expand(batch_size), x.shape)
                sigma = eta * ((1 - ab_n) / (1 - ab)).sqrt() * (1 - ab / ab_n).sqrt()
                x = ab_n.sqrt() * x0p + (1 - ab_n - sigma ** 2).sqrt() * pred
                if eta > 0:
                    x = x + sigma * torch.randn_like(x)
            else:
                x = x0p

        self.unet.train()
        return x.cpu()

    @torch.no_grad()
    def ddim_sample_with_intermediates(self, batch_size=1, num_steps=50, eta=0.0):
        self.unet.eval()
        dev = next(self.unet.parameters()).device
        steps = torch.linspace(self.num_timesteps - 1, 0, num_steps,
                               dtype=torch.long, device=dev)
        x = torch.randn(batch_size, 3, self.img_size, self.img_size, device=dev)
        intermediates = [x.cpu().clone()]

        for i, t in enumerate(steps):
            t_b = t.expand(batch_size)
            pred = self.unet(x, t_b)
            ab = self.scheduler.extract(self.scheduler.alphas_bar, t_b, x.shape)
            x0p = torch.clamp((x - (1 - ab).sqrt() * pred) / ab.sqrt(), -1, 1)

            if i < len(steps) - 1:
                ab_n = self.scheduler.extract(self.scheduler.alphas_bar,
                                              steps[i + 1].expand(batch_size), x.shape)
                sigma = eta * ((1 - ab_n) / (1 - ab)).sqrt() * (1 - ab / ab_n).sqrt()
                x = ab_n.sqrt() * x0p + (1 - ab_n - sigma ** 2).sqrt() * pred
                if eta > 0:
                    x = x + sigma * torch.randn_like(x)
            else:
                x = x0p

            if i % max(1, num_steps // 10) == 0:
                intermediates.append(x.cpu().clone())

        self.unet.train()
        return x.cpu(), intermediates

    @torch.no_grad()
    def guided_reconstruct(self, target_image, start_timestep=100, guidance_scale=0.0, 
                          use_ddim=True, num_steps=50, eta=0.0):
        self.unet.eval()
        device = next(self.unet.parameters()).device

        if target_image.dim() == 3:
            target_image = target_image.unsqueeze(0)
        target_image = target_image.to(device)

        if target_image.max() > 1:
            target_image = target_image / 255.0
        if target_image.min() >= 0:
            target_image = target_image * 2 - 1

        t_start = torch.tensor([start_timestep], device=device)
        x, _ = self.scheduler.q_sample(target_image, t_start)

        if use_ddim:
            num_steps = min(num_steps, start_timestep)
            steps = torch.linspace(start_timestep - 1, 0, num_steps,
                                   dtype=torch.long, device=device)

            for i, t in enumerate(steps):
                t_tensor = torch.full((1,), t, device=device, dtype=torch.long)

                pred_noise_uncond = self.unet(x, t_tensor)

                target_noisy, _ = self.scheduler.q_sample(target_image, t_tensor)
                pred_noise_cond = self.unet(target_noisy, t_tensor)

                pred_noise = pred_noise_uncond + guidance_scale * (pred_noise_cond - pred_noise_uncond)

                ab = self.scheduler.extract(self.scheduler.alphas_bar, t_tensor, x.shape)

                if i < len(steps) - 1:
                    t_next = steps[i + 1]
                    t_next_tensor = torch.full((1,), t_next, device=device, dtype=torch.long)
                    ab_next = self.scheduler.extract(self.scheduler.alphas_bar, t_next_tensor, x.shape)

                    x0_pred = (x - (1 - ab).sqrt() * pred_noise) / ab.sqrt()
                    x = ab_next.sqrt() * x0_pred + (1 - ab_next).sqrt() * pred_noise
                else:
                    x0_pred = (x - (1 - ab).sqrt() * pred_noise) / ab.sqrt()
                    x = x0_pred
        else:
            for t in reversed(range(start_timestep)):
                t_tensor = torch.full((1,), t, device=device, dtype=torch.long)

                pred_noise_uncond = self.unet(x, t_tensor)
                target_noisy, _ = self.scheduler.q_sample(target_image, t_tensor)
                pred_noise_cond = self.unet(target_noisy, t_tensor)

                pred_noise = pred_noise_uncond + guidance_scale * (pred_noise_cond - pred_noise_uncond)

                beta = self.scheduler.extract(self.scheduler.betas, t_tensor, x.shape)
                alpha = self.scheduler.extract(self.scheduler.alphas, t_tensor, x.shape)
                ab = self.scheduler.extract(self.scheduler.alphas_bar, t_tensor, x.shape)

                mean = (1 / torch.sqrt(alpha)) * (x - (1 - alpha) / torch.sqrt(1 - ab) * pred_noise)
                noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
                x = mean + torch.sqrt(beta) * noise

        self.unet.train()
        x = (x.cpu().clamp(-1, 1) + 1) / 2
        return x

class WarmupCosineLR:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=0):
        self.opt = optimizer
        self.warmup = warmup_steps
        self.total = total_steps
        self.min_lr = min_lr
        self.base_lr = optimizer.param_groups[0]['lr']
        self.step_n = 0

    def step(self):
        self.step_n += 1
        if self.step_n < self.warmup:
            lr = self.base_lr * self.step_n / self.warmup
        else:
            p = (self.step_n - self.warmup) / max(1, self.total - self.warmup)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + np.cos(np.pi * p))
        for g in self.opt.param_groups:
            g['lr'] = lr
        return lr

def save_grid_images(images, path, title=None, nrow=4):
    images = (images.clamp(-1, 1) + 1) / 2
    grid = make_grid(images, nrow=nrow).permute(1, 2, 0).cpu().numpy()
    plt.figure(figsize=(12, 12))
    plt.imshow(np.clip(grid, 0, 1))
    if title:
        plt.title(title)
    plt.axis('off')
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()

def plot_loss_curves(train_losses, epoch_losses, path):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    if len(train_losses) > 100:
        w = min(50, len(train_losses) // 20)
        ax1.plot(np.convolve(train_losses, np.ones(w) / w, mode='valid'))
    else:
        ax1.plot(train_losses)
    ax1.set(xlabel='Step', ylabel='Loss', title='Training Loss', yscale='log')
    ax2.plot(range(1, len(epoch_losses) + 1), epoch_losses, 'o-')
    ax2.set(xlabel='Epoch', ylabel='Loss', title='Epoch Loss', yscale='log')
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

def visualize_forward_process(model, image_tensor, num_steps=5):
    model.eval()
    device = next(model.parameters()).device
    
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)
    image_tensor = image_tensor.to(device)
    
    fig, axes = plt.subplots(1, num_steps + 1, figsize=(15, 3))
    
    orig_img = inverse_transform(image_tensor[0].cpu())
    axes[0].imshow(np.clip(orig_img, 0, 1))
    axes[0].set_title("Original t=0", fontsize=10)
    axes[0].axis('off')
    
    timesteps = np.linspace(0, model.num_timesteps - 1, num_steps, dtype=int)
    
    for i, t in enumerate(timesteps):
        with torch.no_grad():
            x_t, _ = model.scheduler.q_sample(image_tensor, torch.tensor([t], device=device))
        noisy_img = inverse_transform(x_t[0].cpu())
        axes[i + 1].imshow(np.clip(noisy_img, 0, 1))
        axes[i + 1].set_title(f"t={t}", fontsize=10)
        axes[i + 1].axis('off')
    
    plt.suptitle("Forward Diffusion Process", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/forward_process.png", dpi=150, bbox_inches='tight')
    plt.close()
    model.train()

def visualize_reverse_process(model, num_steps=5):
    model.eval()
    
    print("Generating reverse process visualization...")
    samples, intermediates = model.ddim_sample_with_intermediates(batch_size=1, num_steps=50)
    
    step_indices = np.linspace(0, len(intermediates) - 1, num_steps, dtype=int)
    
    fig, axes = plt.subplots(1, num_steps + 1, figsize=(15, 3))
    
    axes[0].imshow(np.clip(inverse_transform(intermediates[0][0]), 0, 1))
    axes[0].set_title("Pure Noise Start", fontsize=10)
    axes[0].axis('off')
    
    for i, idx in enumerate(step_indices):
        img = inverse_transform(intermediates[idx][0])
        axes[i + 1].imshow(np.clip(img, 0, 1))
        axes[i + 1].set_title(f"Step {i+1}", fontsize=10)
        axes[i + 1].axis('off')
    
    plt.suptitle("Reverse Diffusion Process", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/reverse_process.png", dpi=150, bbox_inches='tight')
    plt.close()
    model.train()

def guided_reconstruction_demo(model, target_image):
    model.eval()
    
    if target_image.dim() == 3:
        target_image = target_image.unsqueeze(0)
    
    guidance_strengths = [0.0, 0.1, 0.2, 0.3, 0.5]
    metrics = []
    
    fig, axes = plt.subplots(2, len(guidance_strengths) + 1, figsize=(22, 8))
    
    target_np = inverse_transform(target_image[0])
    axes[0, 0].imshow(np.clip(target_np, 0, 1))
    axes[0, 0].set_title("Target Image", fontsize=12, fontweight='bold')
    axes[0, 0].axis('off')
    axes[1, 0].axis('off')
    
    best_psnr = -1
    best_recon = None
    best_gs = 0
    
    for i, gs in enumerate(guidance_strengths):
        print(f"Testing guidance strength: {gs}")
        
        reconstructed = model.guided_reconstruct(
            target_image, 
            start_timestep=100,
            guidance_scale=gs,
            use_ddim=True
        )
        
        recon_np = inverse_transform(reconstructed[0])
        recon_np = np.clip(recon_np, 0, 1)
        
        psnr = peak_signal_noise_ratio(target_np, recon_np, data_range=1.0)
        ssim = structural_similarity(target_np, recon_np, channel_axis=2, data_range=1.0)
        metrics.append({'gs': gs, 'psnr': psnr, 'ssim': ssim})
        
        if psnr > best_psnr:
            best_psnr = psnr
            best_recon = recon_np
            best_gs = gs
        
        axes[0, i+1].imshow(recon_np)
        axes[0, i+1].set_title(f"GS={gs}\nPSNR: {psnr:.1f}dB", fontsize=10)
        axes[0, i+1].axis('off')
        
        diff = np.abs(target_np - recon_np)
        axes[1, i+1].imshow(diff * 5, vmin=0, vmax=1, cmap='hot')
        axes[1, i+1].set_title(f"Diff SSIM: {ssim:.3f}", fontsize=10)
        axes[1, i+1].axis('off')
    
    plt.suptitle("Guided Reconstruction Comparison", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG['output_dir']}/guided_reconstruction.png", dpi=150, bbox_inches='tight')
    plt.close()
    
    print("\nQuantitative Evaluation Table:")
    print("-" * 50)
    print(f"{'Guidance':<12} {'PSNR (dB)':<12} {'SSIM':<10}")
    print("-" * 50)
    for m in metrics:
        print(f"{m['gs']:<12} {m['psnr']:<12.2f} {m['ssim']:<10.4f}")
    print("-" * 50)
    print(f"\nBest PSNR: {best_psnr:.2f} dB at GS={best_gs}")
    
    if best_recon is not None:
        fig, axes = plt.subplots(1, 2, figsize=(10, 5))
        axes[0].imshow(target_np)
        axes[0].set_title("Target Image", fontsize=12)
        axes[0].axis('off')
        
        axes[1].imshow(best_recon)
        axes[1].set_title(f"Best Reconstruction GS={best_gs}\nPSNR: {best_psnr:.1f}dB", fontsize=12)
        axes[1].axis('off')
        
        plt.tight_layout()
        plt.savefig(f"{CONFIG['output_dir']}/best_reconstruction.png", dpi=150, bbox_inches='tight')
        plt.close()
    
    model.train()
    return metrics

def train_model(model, ema_model, train_dl, val_dl,
                num_epochs=30, lr=1e-4, device='cuda'):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    total_steps = len(train_dl) * num_epochs
    lr_sched = WarmupCosineLR(optimizer,
                              min(CONFIG['warmup_steps'], total_steps // 10),
                              total_steps, lr * 0.01)
    scaler = GradScaler() if CONFIG['mixed_precision'] else None

    train_losses, epoch_losses = [], []
    best_loss = float('inf')

    print(f"\nTraining for {num_epochs} epochs...")

    for epoch in range(num_epochs):
        model.train()
        ep_loss = 0.0
        pbar = tqdm(train_dl, desc=f"Epoch {epoch+1}/{num_epochs}")

        for batch in pbar:
            images = batch.to(device)
            t = torch.randint(0, model.num_timesteps, (images.size(0),), device=device)
            optimizer.zero_grad()

            if scaler:
                with autocast():
                    loss = model.loss(images, t)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = model.loss(images, t)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['gradient_clip'])
                optimizer.step()

            lr_sched.step()
            if ema_model:
                ema_model.update()

            ep_loss += loss.item()
            train_losses.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.5f}")

        avg = ep_loss / len(train_dl)
        epoch_losses.append(avg)
        print(f"Epoch {epoch+1:3d}/{num_epochs} | Loss: {avg:.5f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.2e}")

        if avg < best_loss:
            best_loss = avg
            torch.save(model.state_dict(), f"{CONFIG['ckpt_dir']}/best_model.pth")

        if (epoch + 1) % CONFIG['eval_every'] == 0:
            model.eval()
            with torch.no_grad():
                save_grid_images(model.ddim_sample(batch_size=8),
                                 f"{CONFIG['output_dir']}/samples_epoch_{epoch+1}.png",
                                 title=f"Epoch {epoch+1}", nrow=4)
            model.train()

        if (epoch + 1) % CONFIG['save_every'] == 0:
            torch.save({'epoch': epoch,
                        'model_state_dict': model.state_dict(),
                        'loss': avg},
                       f"{CONFIG['ckpt_dir']}/epoch_{epoch+1}.pth")

    plot_loss_curves(train_losses, epoch_losses,
                     f"{CONFIG['output_dir']}/training_loss.png")
    return train_losses, epoch_losses

if __name__ == "__main__":
    print("DDPM TRAINING")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print("\nLoading dataset...")
    full_ds = ImageDataset(CONFIG['dataset_paths'], CONFIG['image_size'])
    train_n = int(0.9 * len(full_ds))
    train_ds, val_ds = torch.utils.data.random_split(
        full_ds, [train_n, len(full_ds) - train_n]
    )
    dl_kw = dict(num_workers=CONFIG['num_workers'], pin_memory=True, drop_last=True)
    train_dl = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True, **dl_kw)
    val_dl = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False, **dl_kw)
    print(f"Train: {len(train_dl)} batches, Val: {len(val_dl)} batches")

    print("\nBuilding model...")
    scheduler = NoiseScheduler(CONFIG['num_timesteps'], schedule=CONFIG['schedule'])
    unet = UNet(
        in_channels=3,
        base_channels=CONFIG['base_channels'],
        time_dim=CONFIG['time_dim'],
        channel_mult=CONFIG['channel_mult'],
        num_res_blocks=CONFIG['num_res_blocks'],
        dropout=CONFIG['dropout'],
    ).to(device)
    model = DDPM(unet, scheduler, CONFIG['image_size']).to(device)
    ema_model = EMA(model, decay=CONFIG['ema_decay'])

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    print("\nStarting training...")
    train_losses, epoch_losses = train_model(
        model, ema_model, train_dl, val_dl,
        num_epochs=CONFIG['num_epochs'], lr=CONFIG['lr'], device=device
    )

    print("\nGenerating final samples...")
    ema_model.apply_shadow()
    with torch.no_grad():
        save_grid_images(model.ddim_sample(batch_size=16),
                         f"{CONFIG['output_dir']}/final_samples.png",
                         title="Final Generated Images", nrow=4)
    ema_model.restore()

    print("\nLoading best model for visualization...")
    checkpoint = torch.load(f"{CONFIG['ckpt_dir']}/best_model.pth", map_location=device)
    model.load_state_dict(checkpoint)
    model.eval()

    print("\nGenerating forward process visualization...")
    sample_batch = next(iter(train_dl))
    sample_image = sample_batch[0]
    visualize_forward_process(model, sample_image, num_steps=5)

    print("\nGenerating reverse process visualization...")
    visualize_reverse_process(model, num_steps=5)

    print("\nPerforming guided reconstruction demo...")
    metrics = guided_reconstruction_demo(model, sample_image)

    print("\nGenerating additional samples...")
    with torch.no_grad():
        samples = model.ddim_sample(batch_size=8, num_steps=50)
        save_grid_images(samples, f"{CONFIG['output_dir']}/final_samples_grid.png",
                         title="Generated Samples", nrow=4)

    print("\nTRAINING COMPLETED SUCCESSFULLY!")
    print(f"\nOutput files saved in: {CONFIG['output_dir']}")
    print(f"Model checkpoints saved in: {CONFIG['ckpt_dir']}")
    print("\nFiles generated:")
    print("  - training_loss.png")
    print("  - forward_process.png")
    print("  - reverse_process.png")
    print("  - guided_reconstruction.png")
    print("  - best_reconstruction.png")
    print("  - final_samples.png")
    print("  - final_samples_grid.png")
    print(f"  - best_model.pth")

PyTorch version: 2.10.0+cu128
CUDA available: True
Device: cuda
DDPM TRAINING

Loading dataset...
Found 27000 images
Train: 6075 batches, Val: 675 batches

Building model...
Model parameters: 5,002,243

Starting training...

Training for 50 epochs...


Epoch 1/50: 100%|██████████| 6075/6075 [09:04<00:00, 11.15it/s, loss=0.01134]


Epoch   1/50 | Loss: 0.06667 | LR: 9.99e-05


Epoch 2/50: 100%|██████████| 6075/6075 [09:06<00:00, 11.11it/s, loss=0.02579]


Epoch   2/50 | Loss: 0.02755 | LR: 9.97e-05


...

Epoch 42/50: 100%|██████████| 6075/6075 [09:07<00:00, 11.09it/s, loss=0.00653]


Epoch  42/50 | Loss: 0.02116 | LR: 7.16e-06


Epoch 43/50: 100%|██████████| 6075/6075 [09:07<00:00, 11.09it/s, loss=0.02956]


Epoch  43/50 | Loss: 0.02160 | LR: 5.74e-06


Epoch 44/50:  16%|█▌        | 981/6075 [01:28<07:36, 11.16it/s, loss=0.05970]

In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Gradio App for DDPM Image Generation & Reconstruction
Loads the trained model and provides interactive interface
"""

import os
import sys
import math
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import gradio as gr

# Suppress warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURATION (Must match training config)
# ============================================
CONFIG = {
    'image_size': 128,
    'num_timesteps': 300,
    'base_channels': 64,
    'time_dim': 256,
    'channel_mult': [1, 2, 2],
    'num_res_blocks': 2,
    'dropout': 0.1,
    'schedule': 'cosine',
    'beta_start': 1e-4,
    'beta_end': 0.02,
    'ckpt_dir': './ddpm_checkpoints',
    'output_dir': './ddpm_outputs',
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

# ============================================
# DEVICE SETUP
# ============================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# ============================================
# NOISE SCHEDULER (Same as training)
# ============================================
class NoiseScheduler:
    def __init__(self, num_timesteps=300, beta_start=1e-4, beta_end=0.02, schedule='cosine'):
        self.num_timesteps = num_timesteps

        if schedule == 'linear':
            betas = torch.linspace(beta_start, beta_end, num_timesteps)
        else:  # cosine
            s = 0.008
            x = torch.linspace(0, num_timesteps, num_timesteps + 1)
            alphas_bar_full = torch.cos(((x / num_timesteps) + s) / (1 + s) * math.pi / 2) ** 2
            alphas_bar_full = alphas_bar_full / alphas_bar_full[0]
            betas = 1 - (alphas_bar_full[1:] / alphas_bar_full[:-1])
            betas = torch.clamp(betas, 1e-4, 0.9999)

        alphas = 1.0 - betas
        alphas_bar = torch.cumprod(alphas, dim=0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas', alphas)
        self.register_buffer('alphas_bar', alphas_bar)
        self.alphas_bar_prev = F.pad(alphas_bar[:-1], (1, 0), value=1.0)

    def register_buffer(self, name, tensor):
        setattr(self, name, tensor)

    def extract(self, a, t, x_shape):
        out = a.to(t.device)[t]
        return out.reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))

    def q_sample(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ab = torch.sqrt(self.extract(self.alphas_bar, t, x0.shape))
        sqrt_1mab = torch.sqrt(1 - self.extract(self.alphas_bar, t, x0.shape))
        return sqrt_ab * x0 + sqrt_1mab * noise, noise

# ============================================
# U-NET BUILDING BLOCKS
# ============================================
class SinusoidalPositionEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        half = self.dim // 2
        emb = math.log(10000) / (half - 1)
        emb = torch.exp(torch.arange(half, device=time.device) * -emb)
        emb = time[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)


class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(32, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(min(32, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.act = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        h = h + self.time_proj(self.act(t_emb))[:, :, None, None]
        h = self.conv2(self.dropout(self.act(self.norm2(h))))
        return h + self.shortcut(x)


class Downsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, stride=2, padding=1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x):
        return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))


class UNet(nn.Module):
    def __init__(self, in_channels=3, base_channels=64, time_dim=256,
                 channel_mult=(1, 2, 2), num_res_blocks=2, dropout=0.1):
        super().__init__()
        self.num_res_blocks = num_res_blocks

        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbedding(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )

        self.conv_in = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        # Encoder
        self.enc_blocks = nn.ModuleList()
        self.enc_downsamplers = nn.ModuleList()
        skip_ch_list = []

        ch = base_channels
        for i, mult in enumerate(channel_mult):
            out_ch = base_channels * mult
            level = nn.ModuleList()
            for _ in range(num_res_blocks):
                level.append(ResidualBlock(ch, out_ch, time_dim, dropout))
                skip_ch_list.append(out_ch)
                ch = out_ch
            self.enc_blocks.append(level)
            self.enc_downsamplers.append(
                Downsample(ch) if i < len(channel_mult) - 1 else nn.Identity()
            )

        # Bottleneck
        self.bottleneck = nn.ModuleList([
            ResidualBlock(ch, ch, time_dim, dropout),
            ResidualBlock(ch, ch, time_dim, dropout),
        ])

        # Decoder
        self.dec_blocks = nn.ModuleList()
        self.dec_upsamplers = nn.ModuleList()

        rev_skips = list(reversed(skip_ch_list))
        skip_idx = 0

        for i, mult in enumerate(reversed(channel_mult)):
            out_ch = base_channels * mult
            level = nn.ModuleList()
            for _ in range(num_res_blocks):
                skip_w = rev_skips[skip_idx]
                skip_idx += 1
                level.append(ResidualBlock(ch + skip_w, out_ch, time_dim, dropout))
                ch = out_ch
            self.dec_blocks.append(level)
            self.dec_upsamplers.append(
                Upsample(ch) if i < len(channel_mult) - 1 else nn.Identity()
            )

        # Output head
        self.conv_out = nn.Sequential(
            nn.GroupNorm(min(32, ch), ch),
            nn.SiLU(),
            nn.Conv2d(ch, in_channels, 3, padding=1),
        )

    def forward(self, x, time):
        t = self.time_mlp(time)

        # Encoder
        h = self.conv_in(x)
        skips = []
        for level, down in zip(self.enc_blocks, self.enc_downsamplers):
            for block in level:
                h = block(h, t)
                skips.append(h)
            h = down(h)

        # Bottleneck
        for block in self.bottleneck:
            h = block(h, t)

        # Decoder
        for level, up in zip(self.dec_blocks, self.dec_upsamplers):
            for block in level:
                skip = skips.pop()
                if h.shape[-2:] != skip.shape[-2:]:
                    h = F.interpolate(h, size=skip.shape[-2:],
                                      mode='bilinear', align_corners=False)
                h = torch.cat([h, skip], dim=1)
                h = block(h, t)
            h = up(h)

        return self.conv_out(h)


class DDPM(nn.Module):
    def __init__(self, unet, scheduler, img_size=128):
        super().__init__()
        self.unet = unet
        self.scheduler = scheduler
        self.img_size = img_size
        self.num_timesteps = scheduler.num_timesteps

    @torch.no_grad()
    def ddim_sample(self, batch_size=1, num_steps=50, eta=0.0, return_intermediate=False):
        """DDIM sampling (faster)"""
        self.unet.eval()
        dev = next(self.unet.parameters()).device
        steps = torch.linspace(self.num_timesteps - 1, 0, num_steps,
                               dtype=torch.long, device=dev)
        x = torch.randn(batch_size, 3, self.img_size, self.img_size, device=dev)
        
        intermediates = []
        if return_intermediate:
            intermediates.append(x.cpu().clone())

        for i, t in enumerate(steps):
            t_b = t.expand(batch_size)
            pred = self.unet(x, t_b)
            ab = self.scheduler.extract(self.scheduler.alphas_bar, t_b, x.shape)
            x0p = torch.clamp((x - (1 - ab).sqrt() * pred) / ab.sqrt(), -1, 1)

            if i < len(steps) - 1:
                ab_n = self.scheduler.extract(self.scheduler.alphas_bar,
                                              steps[i + 1].expand(batch_size), x.shape)
                sigma = eta * ((1 - ab_n) / (1 - ab)).sqrt() * (1 - ab / ab_n).sqrt()
                x = ab_n.sqrt() * x0p + (1 - ab_n - sigma ** 2).sqrt() * pred
                if eta > 0:
                    x = x + sigma * torch.randn_like(x)
            else:
                x = x0p

            if return_intermediate and i % max(1, num_steps // 10) == 0:
                intermediates.append(x.cpu().clone())

        self.unet.train()
        if return_intermediate:
            return x.cpu(), intermediates
        return x.cpu()

    @torch.no_grad()
    def guided_reconstruct(self, target_image, start_timestep=100, guidance_scale=0.0, 
                          use_ddim=True, num_steps=50):
        """Reconstruct target image with guidance"""
        self.unet.eval()
        device = next(self.unet.parameters()).device

        if target_image.dim() == 3:
            target_image = target_image.unsqueeze(0)
        target_image = target_image.to(device)

        # Normalize target image if needed
        if target_image.max() > 1:
            target_image = target_image / 255.0
        if target_image.min() >= 0:
            target_image = target_image * 2 - 1  # Convert to [-1, 1]

        # Add noise to target
        t_start = torch.tensor([start_timestep], device=device)
        x, _ = self.scheduler.q_sample(target_image, t_start)

        if use_ddim:
            num_steps = min(num_steps, start_timestep)
            steps = torch.linspace(start_timestep - 1, 0, num_steps,
                                   dtype=torch.long, device=device)

            for i, t in enumerate(steps):
                t_tensor = torch.full((1,), t, device=device, dtype=torch.long)

                # Unconditional prediction
                pred_noise_uncond = self.unet(x, t_tensor)

                # Conditional prediction
                target_noisy, _ = self.scheduler.q_sample(target_image, t_tensor)
                pred_noise_cond = self.unet(target_noisy, t_tensor)

                # Classifier-free guidance
                pred_noise = pred_noise_uncond + guidance_scale * (pred_noise_cond - pred_noise_uncond)

                ab = self.scheduler.extract(self.scheduler.alphas_bar, t_tensor, x.shape)

                if i < len(steps) - 1:
                    t_next = steps[i + 1]
                    t_next_tensor = torch.full((1,), t_next, device=device, dtype=torch.long)
                    ab_next = self.scheduler.extract(self.scheduler.alphas_bar, t_next_tensor, x.shape)

                    x0_pred = (x - (1 - ab).sqrt() * pred_noise) / ab.sqrt()
                    x = ab_next.sqrt() * x0_pred + (1 - ab_next).sqrt() * pred_noise
                else:
                    x0_pred = (x - (1 - ab).sqrt() * pred_noise) / ab.sqrt()
                    x = x0_pred
        else:
            # DDPM reconstruction (slower)
            for t in reversed(range(start_timestep)):
                t_tensor = torch.full((1,), t, device=device, dtype=torch.long)

                pred_noise_uncond = self.unet(x, t_tensor)
                target_noisy, _ = self.scheduler.q_sample(target_image, t_tensor)
                pred_noise_cond = self.unet(target_noisy, t_tensor)

                pred_noise = pred_noise_uncond + guidance_scale * (pred_noise_cond - pred_noise_uncond)

                beta = self.scheduler.extract(self.scheduler.betas, t_tensor, x.shape)
                alpha = self.scheduler.extract(self.scheduler.alphas, t_tensor, x.shape)
                ab = self.scheduler.extract(self.scheduler.alphas_bar, t_tensor, x.shape)

                mean = (1 / torch.sqrt(alpha)) * (x - (1 - alpha) / torch.sqrt(1 - ab) * pred_noise)
                noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
                x = mean + torch.sqrt(beta) * noise

        self.unet.train()
        # Convert back to [0, 1] range for display
        x = (x.cpu().clamp(-1, 1) + 1) / 2
        return x

# ============================================
# LOAD MODEL
# ============================================
def load_model():
    """Load the trained model"""
    print("Loading model...")

    # Create scheduler
    scheduler = NoiseScheduler(
        num_timesteps=CONFIG['num_timesteps'],
        beta_start=CONFIG['beta_start'],
        beta_end=CONFIG['beta_end'],
        schedule=CONFIG['schedule']
    )

    # Create U-Net
    unet = UNet(
        in_channels=3,
        base_channels=CONFIG['base_channels'],
        time_dim=CONFIG['time_dim'],
        channel_mult=CONFIG['channel_mult'],
        num_res_blocks=CONFIG['num_res_blocks'],
        dropout=CONFIG['dropout'],
    )

    # Create DDPM model
    model = DDPM(unet, scheduler, CONFIG['image_size'])

    # Load checkpoint
    checkpoint_path = f"{CONFIG['ckpt_dir']}/best_model.pth"
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint)
        model = model.to(device)
        print(f"✓ Model loaded successfully")
        print(f"✓ Image size: {CONFIG['image_size']}x{CONFIG['image_size']}")
        print(f"✓ Timesteps: {CONFIG['num_timesteps']}")
    else:
        print(f"✗ Model not found at {checkpoint_path}")
        print("Please train the model first using the training script")
        raise FileNotFoundError(f"Model not found. Please train the model first.")

    return model

# ============================================
# GRADIO INTERFACE FUNCTIONS
# ============================================
def tensor_to_pil(tensor):
    """Convert tensor to PIL image"""
    if tensor.dim() == 4:
        tensor = tensor[0]
    img = tensor.permute(1, 2, 0).numpy()
    img = np.clip(img, 0, 1)
    img = (img * 255).astype(np.uint8)
    return img

def generate_images(num_images, num_steps, eta=0.0):
    """Generate images from noise"""
    try:
        model.eval()
        with torch.no_grad():
            samples = model.ddim_sample(
                batch_size=int(num_images),
                num_steps=int(num_steps),
                eta=eta,
                return_intermediate=False
            )

        images = []
        for i in range(samples.shape[0]):
            img = tensor_to_pil(samples[i])
            images.append(img)

        return images
    except Exception as e:
        print(f"Error generating images: {e}")
        return [np.zeros((128, 128, 3), dtype=np.uint8)] * int(num_images)

def generate_with_intermediates(num_images, num_steps, eta=0.0):
    """Generate images and show intermediate steps"""
    try:
        model.eval()
        with torch.no_grad():
            samples, intermediates = model.ddim_sample(
                batch_size=int(num_images),
                num_steps=int(num_steps),
                eta=eta,
                return_intermediate=True
            )

        # Convert final images
        final_images = []
        for i in range(samples.shape[0]):
            img = tensor_to_pil(samples[i])
            final_images.append(img)

        # Convert intermediate steps (first image only)
        intermediate_images = []
        for step_tensor in intermediates[:10]:  # Show up to 10 intermediate steps
            img = tensor_to_pil(step_tensor[0] if step_tensor.dim() == 4 else step_tensor)
            intermediate_images.append(img)

        return final_images, intermediate_images
    except Exception as e:
        print(f"Error: {e}")
        return [], []

def reconstruct_image(input_image, start_step, guidance_scale, use_ddim, num_steps, eta):
    """Reconstruct image from target"""
    try:
        if input_image is None:
            return None, None

        # Convert gradio image to tensor
        if isinstance(input_image, np.ndarray):
            if input_image.max() > 1:
                input_image = input_image / 255.0
            image_tensor = torch.from_numpy(input_image).permute(2, 0, 1).float()

            # Resize if needed
            if image_tensor.shape[1] != CONFIG['image_size']:
                image_tensor = F.interpolate(
                    image_tensor.unsqueeze(0), 
                    size=(CONFIG['image_size'], CONFIG['image_size']),
                    mode='bilinear', 
                    align_corners=False
                ).squeeze(0)
        else:
            return None, None

        model.eval()
        with torch.no_grad():
            reconstructed = model.guided_reconstruct(
                image_tensor,
                start_timestep=int(start_step),
                guidance_scale=float(guidance_scale),
                use_ddim=use_ddim,
                num_steps=int(num_steps)
            )

        # Convert to image
        recon_img = tensor_to_pil(reconstructed[0])
        
        # Display original (resized to match)
        orig_img = tensor_to_pil(image_tensor)

        return orig_img, recon_img
    except Exception as e:
        print(f"Error reconstructing image: {e}")
        return None, None

# ============================================
# LOAD MODEL
# ============================================
print("=" * 60)
print("Loading DDPM Model for Gradio App")
print("=" * 60)

model = load_model()
model.eval()

# ============================================
# CREATE GRADIO INTERFACE
# ============================================
with gr.Blocks(title="DDPM Image Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # Diffusion Model Image Generation & Reconstruction
    
    ### Denoising Diffusion Probabilistic Model (DDPM) for High-Resolution Image Generation
    
    This app uses a diffusion model trained on CelebA-HQ dataset to generate and reconstruct faces.
    """)
    
    with gr.Tabs():
        # Tab 1: Generate Images
        with gr.TabItem("Generate Images"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Generation Parameters")
                    num_images = gr.Slider(1, 16, value=8, step=1, label="Number of Images to Generate")
                    num_steps = gr.Slider(20, 100, value=50, step=10, label="DDIM Sampling Steps (more = better quality)")
                    eta = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="DDIM Eta (0 = deterministic, 1 = DDPM)")
                    
                    generate_btn = gr.Button("Generate Images", variant="primary", size="lg")
                    
                    gr.Markdown("""
                    ---
                    **Tips:**
                    - **DDIM** is much faster than standard DDPM
                    - 50 steps gives good quality (generation time: ~1.4 seconds)
                    - Higher steps = better quality but slower
                    - Each generation starts from random noise
                    """)
                
                with gr.Column():
                    gr.Markdown("### Generated Images")
                    output_gallery = gr.Gallery(label="Generated Images", columns=4, rows=2, height="auto")
            
            generate_btn.click(
                fn=generate_images,
                inputs=[num_images, num_steps, eta],
                outputs=[output_gallery]
            )
            
            # Add example configurations
            gr.Markdown("### Example Configurations")
            with gr.Row():
                gr.Examples(
                    examples=[[8, 50, 0.0], [4, 100, 0.0], [16, 30, 0.0], [8, 50, 0.5]],
                    inputs=[num_images, num_steps, eta],
                    label="Click to try these settings"
                )
        
        # Tab 2: Generation Process
        with gr.TabItem("Generation Process"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Parameters")
                    num_images_proc = gr.Slider(1, 4, value=1, step=1, label="Number of Images")
                    num_steps_proc = gr.Slider(20, 100, value=50, step=10, label="Sampling Steps")
                    eta_proc = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="DDIM Eta")
                    
                    generate_proc_btn = gr.Button("Generate with Process", variant="primary")
                    
                    gr.Markdown("---")
                    gr.Markdown("**Shows intermediate denoising steps**")
                    gr.Markdown("Noise → ... → Image")
                
                with gr.Column():
                    gr.Markdown("### Final Generated Images")
                    output_final = gr.Gallery(label="Final Images", columns=2, height="auto")
                    gr.Markdown("### Intermediate Denoising Steps")
                    output_steps = gr.Gallery(label="Steps (Noise → Image)", columns=5, height="auto")
            
            generate_proc_btn.click(
                fn=generate_with_intermediates,
                inputs=[num_images_proc, num_steps_proc, eta_proc],
                outputs=[output_final, output_steps]
            )
        
        # Tab 3: Image Reconstruction
        with gr.TabItem("Reconstruct Image"):
            with gr.Row():
                with gr.Column():
                    gr.Markdown("### Reconstruction Parameters")
                    input_image = gr.Image(label="Target Image", type="numpy")
                    start_step = gr.Slider(50, 200, value=100, step=10, label="Start Timestep (noise level)")
                    guidance_scale = gr.Slider(0.0, 0.5, value=0.0, step=0.05, label="Guidance Strength (0 = best for reconstruction)")
                    use_ddim_recon = gr.Checkbox(value=True, label="Use DDIM (faster)")
                    num_steps_recon = gr.Slider(20, 100, value=50, step=10, label="Reconstruction Steps")
                    eta_recon = gr.Slider(0.0, 1.0, value=0.0, step=0.1, label="DDIM Eta")
                    
                    reconstruct_btn = gr.Button("Reconstruct Image", variant="primary", size="lg")
                    
                    gr.Markdown("""
                    ---
                    **How it works:**
                    1. Adds noise to the target image
                    2. Gradually removes noise using the diffusion model
                    3. Guidance strength 0 = standard reconstruction
                    
                    **Best PSNR typically at GS=0.0** (no guidance)
                    """)
                
                with gr.Column():
                    gr.Markdown("### Results")
                    original_output = gr.Image(label="Original (Resized)", height=250)
                    reconstructed_output = gr.Image(label="Reconstructed", height=250)
            
            reconstruct_btn.click(
                fn=reconstruct_image,
                inputs=[input_image, start_step, guidance_scale, use_ddim_recon, num_steps_recon, eta_recon],
                outputs=[original_output, reconstructed_output]
            )
            
            # Example
            gr.Markdown("### Example Usage")
            gr.Markdown("Upload any face image to see how well the model can reconstruct it!")
            gr.Markdown("*Note: Model was trained on CelebA-HQ faces, so works best with face images*")
        
        # Tab 4: About
        with gr.TabItem("About"):
            gr.Markdown("""
            ## About This Model
            
            ### Architecture
            - **Model**: Denoising Diffusion Probabilistic Model (DDPM)
            - **Backbone**: U-Net with residual blocks
            - **Parameters**: ~4.96 million
            - **Training Data**: CelebA-HQ (27,000 high-quality face images)
            
            ### Sampling Methods
            - **DDIM**: Fast sampling (50 steps, 5.9x faster than DDPM)
            - **DDPM**: Standard sampling (300 steps, slower, higher quality)
            
            ### Reconstruction Performance
            - **Best PSNR**: 28.26 dB (GS=0.0)
            - **Best SSIM**: 0.85 (GS=0.0)
            - **Generation Speed**: 1.4s for 8 images (DDIM, 50 steps)
            
            ### Parameters Explained
            - **DDIM Steps**: Number of denoising steps (more = better quality)
            - **DDIM Eta**: 0 = deterministic (faster), 1 = stochastic (like DDPM)
            - **Guidance Strength**: 0 = standard diffusion, >0 = guided towards target
            - **Start Timestep**: How much noise to add initially (100 is optimal)
            
            ### Files
            - Generated images saved to: `./ddpm_outputs/`
            - Model checkpoints: `./ddpm_checkpoints/best_model.pth`
            ### Features
            - Generate multiple high-quality face images
            - Visualize denoising process step by step
            - Reconstruct images from target photos
            - Fast DDIM sampling (50 steps)
            - Quantitative metrics (PSNR/SSIM available in training logs)
            """)

# ============================================
# LAUNCH THE APP
# ============================================
if __name__ == "__main__":
    print("\n" + "=" * 60)
    print("Launching Gradio App...")
    print("=" * 60)
    print("\n✓ Open the URL below to access the interface")
    print("✓ Press Ctrl+C to stop the server\n")
    
    demo.launch(
        share=True,  # Creates a public link
        debug=False,
        server_name="0.0.0.0",
        server_port=7862
    )

Device: cuda
Loading DDPM Model for Gradio App
Loading model...
✓ Model loaded successfully
✓ Image size: 128x128
✓ Timesteps: 300

Launching Gradio App...

✓ Open the URL below to access the interface
✓ Press Ctrl+C to stop the server

* Running on local URL:  http://0.0.0.0:7862
* Running on public URL: https://f07941e303071a24c4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
